# Knowledge base setup

**Notebook 3 of 5.** Builds the full Foundry IQ stack for the Contoso multi-agent lab:
three Knowledge Sources, three Knowledge Bases (HR, Marketing, Products), validates
each KB with a representative query, then creates three MCP connections.

## Prerequisites

1. **Run `11-02-index-and-ingest.ipynb`** - the three Contoso indexes must exist and
   contain 8 documents each.
2. **Python environment** - run `uv sync` from the repo root; select the `.venv` kernel.
3. **Azure CLI** - run `az login` before executing cells.

## Imports and configuration

In [1]:
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

repo_root = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent
load_dotenv(repo_root / '.env', override=True)

# ── Resource names ────────────────────────────────────────────────────────────
DOMAINS = ['hr', 'marketing', 'products']

KS_NAMES  = {d: f'contoso-ks-{d}'  for d in DOMAINS}
KB_NAMES  = {d: f'contoso-kb-{d}'  for d in DOMAINS}
MCP_NAMES = {d: f'contoso-mcp-{d}' for d in DOMAINS}

# ── Environment variables ─────────────────────────────────────────────────────
search_endpoint  = os.environ['CONTOSO_SEARCH_ENDPOINT']
project_endpoint = os.environ['CONTOSO_FOUNDRY_PROJECT_ENDPOINT']
apim_connection  = os.environ.get('CONTOSO_APIM_CONNECTION', 'contoso-apim-connection')
gateway_url      = os.environ['GATEWAY_URL']
gateway_key      = os.environ['CONTOSO_GATEWAY_KEY']
chat_model       = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

# AzureOpenAIVectorizerParameters.resource_url must be root APIM URL (no /openai)
apim_base = gateway_url.rstrip('/').removesuffix('/openai')

# Azure subscription and resource group for MCP connection management API
subscription_id = os.environ.get('AZURE_SUBSCRIPTION_ID') or subprocess.run(
    'az account show --query id -o tsv', shell=True, capture_output=True, text=True
).stdout.strip()

resource_group = os.environ['CONTOSO_RESOURCE_GROUP']
project_name   = os.environ['CONTOSO_FOUNDRY_PROJECT']
account_name   = os.environ['MULTI_ACCOUNT']

print(f'Search endpoint  : {search_endpoint}')
print(f'Project endpoint : {project_endpoint}')
print(f'Account name     : {account_name}')
print(f'Project name     : {project_name}')
print(f'APIM base        : {apim_base}')
print(f'Subscription     : {subscription_id}')
print(f'Resource group   : {resource_group}')

Search endpoint  : https://contoso-search-n5d3ja.search.windows.net
Project endpoint : https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/contoso-project
Account name     : aif-spoke-multi-c2676f
Project name     : contoso-project
APIM base        : https://apim-foundry-c2676f.azure-api.net
Subscription     : 00000000-0000-0000-0000-000000000000
Resource group   : rg-foundry-multi-c2676f


## Create clients

In [2]:
from azure.search.documents.indexes import SearchIndexClient
from azure.ai.projects import AIProjectClient

credential     = DefaultAzureCredential()
index_client   = SearchIndexClient(endpoint=search_endpoint, credential=credential)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)

print('SearchIndexClient : ready')
print('AIProjectClient   : ready')

SearchIndexClient : ready
AIProjectClient   : ready


---
## Phase 1: Knowledge Sources

A **Knowledge Source** registers an existing search index as a named, reusable data
source that KBs can reference. Each domain gets its own KS pointing at its index and
semantic configuration.

In [ ]:
# Verify all three indexes exist before creating knowledge sources
existing_indexes = {i.name for i in index_client.list_indexes()}
for domain in DOMAINS:
    index_name = f'contoso-{domain}'
    assert index_name in existing_indexes, (
        f"Index '{index_name}' not found - run 11-02-index-and-ingest.ipynb first."
    )
    print(f"Index '{index_name}' found.")

In [4]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    SearchIndexFieldReference,
)

DOMAIN_DESCRIPTIONS = {
    'hr':        'Contoso HR policies, benefits, and employee programs (8 documents)',
    'marketing': 'Contoso marketing campaigns, brand guidelines, and strategy (8 documents)',
    'products':  'Contoso product specifications and features (8 documents)',
}

for domain in DOMAINS:
    ks = SearchIndexKnowledgeSource(
        name=KS_NAMES[domain],
        description=DOMAIN_DESCRIPTIONS[domain],
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=f'contoso-{domain}',
            semantic_configuration_name=f'contoso-{domain}-semantic',
            source_data_fields=[
                SearchIndexFieldReference(name='id'),
                SearchIndexFieldReference(name='title'),
                SearchIndexFieldReference(name='category'),
            ],
        ),
    )
    index_client.create_or_update_knowledge_source(ks)
    print(f"Knowledge source '{KS_NAMES[domain]}' created.")

Knowledge source 'contoso-ks-hr' created.
Knowledge source 'contoso-ks-marketing' created.
Knowledge source 'contoso-ks-products' created.


---
## Phase 2: Knowledge Bases

Each KB uses `answerSynthesis` output mode with `low` reasoning effort: one LLM pass
synthesises a natural-language answer from the retrieved documents. This mode is
appropriate for the specialist agent use case where grounded, readable answers are
required. Standard SKU search is required for this mode.

In [5]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
    AzureOpenAIVectorizerParameters,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalLowReasoningEffort,
)

RETRIEVAL_INSTRUCTIONS = {
    'hr': (
        'Answer questions about Contoso HR policies, benefits, and employee programs. '
        'Cite document titles in your answers.'
    ),
    'marketing': (
        'Answer questions about Contoso marketing campaigns, brand guidelines, and strategy. '
        'Cite document titles in your answers.'
    ),
    'products': (
        'Answer questions about Contoso product specifications, features, and pricing. '
        'Cite product names and document titles in your answers.'
    ),
}

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=apim_base,
    deployment_name=chat_model,
    model_name=chat_model,
    api_key=gateway_key,
)

for domain in DOMAINS:
    kb = KnowledgeBase(
        name=KB_NAMES[domain],
        description=f'Contoso {domain.title()} KB - answerSynthesis, low reasoning effort.',
        retrieval_instructions=RETRIEVAL_INSTRUCTIONS[domain],
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
        knowledge_sources=[KnowledgeSourceReference(name=KS_NAMES[domain])],
        models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    )
    index_client.create_or_update_knowledge_base(kb)
    print(f"Knowledge base '{KB_NAMES[domain]}' created (answerSynthesis, low effort).")

Knowledge base 'contoso-kb-hr' created (answerSynthesis, low effort).
Knowledge base 'contoso-kb-marketing' created (answerSynthesis, low effort).
Knowledge base 'contoso-kb-products' created (answerSynthesis, low effort).


---
## Phase 3: KB Validation

Validate each KB with a representative query before wiring to agents.
Uses the `messages` request pattern (required for `answerSynthesis` mode).

In [6]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)
from display_helpers import show_kb_result_detail

VALIDATION_QUERIES = {
    'hr':        'What is the Contoso remote work policy?',
    'marketing': 'What are the key elements of the Contoso brand guidelines?',
    'products':  'What are the specifications of the ContosoBook Pro?',
}


def make_kb_request(query: str, ks_name: str) -> KnowledgeBaseRetrievalRequest:
    """Build a messages-based retrieval request for answerSynthesis KBs."""
    return KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role='user',
                content=[KnowledgeBaseMessageTextContent(text=query)],
            )
        ],
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=ks_name,
                include_references=True,
                include_reference_source_data=True,
            )
        ],
        include_activity=True,
    )


def refs_to_list(references):
    return [
        {
            'title':       (r.source_data or {}).get('title', r.id),
            'id':          str(r.id),
            'source_data': r.source_data or {},
        }
        for r in (references or [])
    ]


print('KB validation helpers ready')

KB validation helpers ready


In [7]:
for domain in DOMAINS:
    query   = VALIDATION_QUERIES[domain]
    kb_name = KB_NAMES[domain]
    ks_name = KS_NAMES[domain]

    kb_client = KnowledgeBaseRetrievalClient(
        endpoint=search_endpoint,
        knowledge_base_name=kb_name,
        credential=credential,
    )

    result = kb_client.retrieve(make_kb_request(query, ks_name))
    show_kb_result_detail(
        {
            'response':   [{'content': [{'text': result.response[0].content[0].text}]}],
            'references': refs_to_list(result.references),
        },
        label=f'{kb_name} - validation query: "{query}"',
    )
    print()

**contoso-kb-hr - validation query: "What is the Contoso remote work policy?"**

Contoso Corporation's Remote Work Policy allows employees to work from approved locations outside the primary office with manager approval and role eligibility. Employees must have a secure, distraction-free workspace with reliable internet of at least 25 Mbps. Core working hours are from 10:00 AM to 3:00 PM local time for synchronous collaboration. Remote workers must attend all scheduled team meetings via video and keep their calendars updated. Equipment such as laptops and peripherals is provided by Contoso IT, and employees are responsible for their care and return upon separation. Personal use of company equipment is prohibited. Data security requirements apply equally to remote and in-office employees, including mandatory VPN use for internal systems access. Employees may work remotely up to five days per week or on a hybrid basis with manager approval. New employees must be on-site for their first 30 days for onboarding. Remote work privileges can be revoked if performance, collaboration, or security standards are not met. Requests to work remotely from international locations require prior HR and Legal approval due to tax and compliance issues. The policy is reviewed annually and updated as needed [ref_id:0].

**Sources:**
- Remote Work Policy
- Paid Time Off Policy
- Performance Review Process
- Employee Benefits Guide
- Professional Development Program

**contoso-kb-marketing - validation query: "What are the key elements of the Contoso brand guidelines?"**

The key elements of the Contoso brand guidelines include: the primary logo is the Contoso wordmark in Contoso Blue (#0078D4), which must never be stretched, recolored outside approved palettes, or placed on competing backgrounds; a minimum clear space equal to the height of the letter 'C' in the wordmark must be maintained on all sides. The secondary color palette consists of Contoso Blue, Contoso Teal (#00B4D8), Contoso Grey (#505050), and White (#FFFFFF), with accent colors Contoso Orange (#FF6B35) and Contoso Green (#107C10) used sparingly for calls to action and status indicators. Typography uses Segoe UI Semibold for headings and Segoe UI Regular for body copy, with minimum sizes specified for print and digital. Photography style is natural, warm, and people-first, avoiding staged or inauthentic stock imagery. Product photography must follow specific studio specifications. Iconography uses the Fluent UI icon library at 24px baseline for digital surfaces. All new marketing materials require Brand team review with a five-business-day SLA. Approved templates are available on the Brand Hub on SharePoint, and brand questions should be directed to brand@contoso.com. These details are outlined in the "Contoso Brand Guidelines" document [ref_id:0].

**Sources:**
- Contoso Brand Guidelines
- Influencer Partnership Program
- SEO Strategy Guide 2025
- Social Media Playbook
- Content Calendar 2025
- Competitor Analysis Report Q1 2025

**contoso-kb-products - validation query: "What are the specifications of the ContosoBook Pro?"**

The ContosoBook Pro is Contoso Corporation's flagship business laptop designed for professionals requiring performance, portability, and AI-accelerated productivity. It is powered by the Intel Core Ultra 7 Series 2 processor with Contoso ContextSense NPU, delivering up to 48 TOPS of AI compute for features like real-time transcription, background noise cancellation, and intelligent power management. The laptop features a 14-inch 2880×1800 OLED display with a 120Hz refresh rate, 400 nits brightness, and 100% DCI-P3 color coverage. Battery life reaches up to 18 hours on video playback under standard conditions, supported by 80W Thunderbolt 4 charging. Memory options include 16GB or 32GB LPDDR5x, and storage options are 512GB, 1TB, or 2TB NVMe PCIe Gen 4. Connectivity includes two Thunderbolt 4 ports, two USB-A 3.2 Gen 2 ports, HDMI 2.1, SD card reader, 3.5mm combo audio jack, and Wi-Fi 7 with Bluetooth 5.4. The chassis is made from aircraft-grade aluminum, weighing 1.28 kg. The keyboard has a full-size island layout with 1.5mm key travel, backlit keys, a fingerprint reader integrated into the power button, and an IR camera for Windows Hello facial recognition. Dimensions are 311mm × 222mm × 14.8mm. Available colors are Platinum Silver and Midnight Black. It ships with Windows 11 Pro and includes a one-year Contoso Premier Support subscription. The retail price starts at $1,299 USD, with a three-year on-site warranty available. SKU: CBP-14-001 [ref_id:0].

**Sources:**
- ContosoBook Pro
- ContosoVision AR Glasses
- ContosoTab
- ContosoEarBuds Pro

---
## Phase 4: MCP Connections

Each KB automatically exposes an MCP endpoint:
```
{search_endpoint}/knowledgebases/{kb_name}/mcp?api-version=2025-11-01-preview
```
A `RemoteTool` connection registered on `contoso-project` allows the Agent Framework
to call each KB via MCP using the project's managed identity.

In [8]:
from iq_helpers import create_mcp_connection
from display_helpers import show_success, show_error

for domain in DOMAINS:
    result = create_mcp_connection(
        subscription_id=subscription_id,
        resource_group=resource_group,
        account_name=account_name,
        project_name=project_name,
        connection_name=MCP_NAMES[domain],
        search_endpoint=search_endpoint,
        kb_name=KB_NAMES[domain],
    )
    if 'error' not in result:
        show_success(f"MCP connection '{MCP_NAMES[domain]}' → '{KB_NAMES[domain]}'")
    else:
        show_error(f"{MCP_NAMES[domain]}: {result}")

### ✅ MCP connection 'contoso-mcp-hr' → 'contoso-kb-hr'

### ✅ MCP connection 'contoso-mcp-marketing' → 'contoso-kb-marketing'

### ✅ MCP connection 'contoso-mcp-products' → 'contoso-kb-products'

---
## Done

The Foundry IQ multi-agent stack is fully provisioned:

| Resource | Name | Mode |
|----------|------|------|
| Knowledge Source | `contoso-ks-hr` | - |
| Knowledge Source | `contoso-ks-marketing` | - |
| Knowledge Source | `contoso-ks-products` | - |
| Knowledge Base | `contoso-kb-hr` | `answerSynthesis` / low effort |
| Knowledge Base | `contoso-kb-marketing` | `answerSynthesis` / low effort |
| Knowledge Base | `contoso-kb-products` | `answerSynthesis` / low effort |
| MCP Connection | `contoso-mcp-hr` | `RemoteTool` → `contoso-kb-hr` |
| MCP Connection | `contoso-mcp-marketing` | `RemoteTool` → `contoso-kb-marketing` |
| MCP Connection | `contoso-mcp-products` | `RemoteTool` → `contoso-kb-products` |

**Next step:** run `11-04-multi-agent-setup.ipynb` to instantiate the Agent Framework
agents and validate routing.